# ROME causal tracing and rank-one editing reproduction

This notebook implements issue #109 as an evidence-producing, fail-closed reproduction. It compares TDHook with the released ROME implementation for causal tracing, carries the released rank-one update through a temporary TDHook parameter intervention, and evaluates a preregistered CounterFact slice. Creating or executing a smoke version of this notebook does not constitute a successful reproduction; only the declared gates in the emitted summary do.

Resource boundary: GPT-2 XL requires CUDA, about 7 GB of downloads, and approximately 16 GB of VRAM. The 100-case edit evaluation is intentionally excluded from CI. Start an environment with `uv run --group notebooks --group scripts jupyter lab` and install the pinned official repository with `uv pip install -e git+https://github.com/kmeng01/rome.git@0874014cd9837e4365f3e6f3c71400ef11509e04#egg=rome`.

## Frozen intervention, controls, metrics, and gates

The intervention is ROME's subject-embedding corruption followed by clean-state restoration at one token/layer, then its layer-17 GPT-2 XL rank-one MLP edit. Tracing controls are clean, corrupted, and a seeded random-layer restoration. Editing controls are pre-edit/no-edit and automatic restoration of every temporary weight change. Metrics are answer probability for tracing and official CounterFact negative-log-likelihood orderings for rewrite efficacy, paraphrase generalization, and neighborhood specificity. The protocol JSON fixes parity tolerance, localization ordering, confidence intervals, revisions, seeds, and checksums before any result is observed.

In [ ]:
from hashlib import sha256
import json
from pathlib import Path
import random
import sys

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

notebook_relative_dir = Path("docs/source/notebooks/tutorials")
notebook_dir = next(
    (
        root / notebook_relative_dir
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / notebook_relative_dir).is_dir()
    ),
    Path.cwd() if Path.cwd().name == "tutorials" else None,
)
if notebook_dir is None:
    raise FileNotFoundError(f"Could not locate {notebook_relative_dir} from {Path.cwd()} or its parents")
sys.path.insert(0, str(notebook_dir))

from rome_reproduction import (  # noqa: E402
    CausalTraceConfig,
    causal_trace_grid,
    causal_trace_window_grid,
    config_dict,
    parity_report,
    summarize_counterfact,
    temporary_rank_one_edit,
    write_json,
)

PROTOCOL_PATH = notebook_dir.parent / "assets/rome-issue-109-protocol.json"
protocol = json.loads(PROTOCOL_PATH.read_text())
assert protocol["artifact_status"] == "not_run"
random.seed(protocol["counterfact"]["seed"])
np.random.seed(protocol["counterfact"]["seed"])
torch.manual_seed(protocol["counterfact"]["seed"])

In [ ]:
# Fail closed on every external scientific asset.
def download_verified(url, destination, expected_sha256):
    import urllib.request

    destination = Path(destination)
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)
    actual = sha256(destination.read_bytes()).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f"checksum mismatch for {destination}: {actual}")
    return destination


data_dir = Path("rome-issue-109-data")
data_dir.mkdir(exist_ok=True)
known_path = download_verified(
    protocol["provenance"]["known_1000_url"], data_dir / "known_1000.json", protocol["provenance"]["known_1000_sha256"]
)
counterfact_path = download_verified(
    protocol["provenance"]["counterfact_url"],
    data_dir / "counterfact.json",
    protocol["provenance"]["counterfact_sha256"],
)
knowns = json.loads(known_path.read_text())
counterfact = json.loads(counterfact_path.read_text())
cases = counterfact[: protocol["counterfact"]["number_cases"]]
assert [case["case_id"] for case in cases] == list(range(100))

In [ ]:
# Load the exact model/tokenizer snapshot and the exact official ROME source.
from experiments.causal_trace import (
    collect_embedding_std,
    find_token_range,
    make_inputs,
    predict_from_input,
    trace_important_states,
    trace_important_window,
    trace_with_patch,
)
from experiments.py.eval_utils_counterfact import compute_rewrite_quality_counterfact
from rome import ROMEHyperParams
from rome.rome_main import execute_rome, upd_matrix_match_shape
from util import nethook

model_snapshot = protocol["provenance"]["model"]
model_revision = protocol["provenance"]["model_revision"]
if not torch.cuda.is_available():
    raise RuntimeError("This exact GPT-2 XL reproduction requires a CUDA accelerator with about 16 GB memory.")
tokenizer = AutoTokenizer.from_pretrained(model_snapshot, revision=model_revision)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_snapshot, revision=model_revision).eval().cuda()
for parameter in model.parameters():
    parameter.requires_grad_(False)

assert protocol["provenance"]["rome_revision"] == "0874014cd9837e4365f3e6f3c71400ef11509e04"

## Fixed-case causal-tracing parity

The official run defines the answer token, subject token span, `RandomState(1)` corruption, ten corrupted copies, and layer names. TDHook receives those identical inputs. The full token-by-layer tensors are retained per case.

In [ ]:
noise = 3.0 * collect_embedding_std(
    type("MT", (), {"model": model, "tokenizer": tokenizer})(), [item["subject"] for item in knowns]
)
trace_config = CausalTraceConfig(samples=10, noise_seed=1, noise_level=noise, replace_noise=False)
layer_paths = [f"transformer.h.{layer}" for layer in range(model.config.n_layer)]
trace_cases = []
for case_id in protocol["tracing"]["parity_case_ids"]:
    case = knowns[case_id]
    inputs = make_inputs(tokenizer, [case["prompt"]] * (trace_config.samples + 1))
    answer_tokens, clean_probabilities = predict_from_input(model, inputs)
    answer_token = int(answer_tokens[0])
    clean_probability = float(clean_probabilities[0])
    assert tokenizer.decode([answer_token]).strip() == case["attribute"]
    subject_range = find_token_range(tokenizer, inputs["input_ids"][0], case["subject"])
    official = trace_important_states(model, model.config.n_layer, inputs, subject_range, answer_token, noise=noise)
    official_mlp = trace_important_window(
        model, model.config.n_layer, inputs, subject_range, answer_token, kind="mlp", window=10, noise=noise
    )
    tdhook_scores, pass_budget = causal_trace_grid(
        model, inputs, answer_token, subject_range, layer_paths, config=trace_config
    )
    tdhook_mlp, mlp_pass_budget = causal_trace_window_grid(
        model, inputs, answer_token, subject_range, layer_paths, component="mlp", window=10, config=trace_config
    )
    parity = parity_report(
        tdhook_scores, official, atol=protocol["tracing"]["parity_atol"], rtol=protocol["tracing"]["parity_rtol"]
    )
    mlp_parity = parity_report(
        tdhook_mlp, official_mlp, atol=protocol["tracing"]["parity_atol"], rtol=protocol["tracing"]["parity_rtol"]
    )
    corrupted = trace_with_patch(model, inputs, [], answer_token, subject_range, noise=noise).item()
    random_layer = int(np.random.default_rng(109 + case_id).integers(model.config.n_layer))
    random_layer_probability = trace_with_patch(
        model, inputs, [(subject_range[1] - 1, layer_paths[random_layer])], answer_token, subject_range, noise=noise
    ).item()
    total_passes = pass_budget["model_passes"] + mlp_pass_budget["model_passes"] + 3
    trace_cases.append(
        {
            "case_id": case_id,
            "subject_range": list(subject_range),
            "answer_token": answer_token,
            "noise": noise,
            "clean_answer": case["attribute"],
            "clean_probability": clean_probability,
            "corrupted_probability": corrupted,
            "random_layer_control": {"layer": random_layer, "probability": random_layer_probability},
            "parity": parity,
            "mlp_parity": mlp_parity,
            "pass_budget": {**pass_budget, "model_passes_including_controls": total_passes},
            "mlp_pass_budget": mlp_pass_budget,
            "tdhook_scores": tdhook_scores.cpu().tolist(),
            "official_scores": official.cpu().tolist(),
            "tdhook_mlp_scores": tdhook_mlp.cpu().tolist(),
            "official_mlp_scores": official_mlp.cpu().tolist(),
        }
    )
    write_json(data_dir / f"trace-case-{case_id}.json", trace_cases[-1])
assert all(case["parity"]["matches"] and case["mlp_parity"]["matches"] for case in trace_cases)

## Aggregate localization heatmap

After fixed-case parity passes, the preregistered aggregate uses every `known_1000` case. Token rows are aligned at the last subject token (offset zero), preserving five positions on either side with missing positions represented as NaN. This produces a comparable 11-by-48 MLP restoration heatmap and does not silently discard cases.

In [ ]:
offsets = protocol["tracing"]["aggregate_token_offsets_from_subject_last"]
aggregate_cases = []
aggregate_passes = 0
for case in knowns:
    inputs = make_inputs(tokenizer, [case["prompt"]] * (trace_config.samples + 1))
    answer_tokens, _ = predict_from_input(model, inputs)
    answer_token = int(answer_tokens[0])
    if tokenizer.decode([answer_token]).strip() != case["attribute"]:
        raise RuntimeError(f"known_1000 case {case['known_id']} is not known by the pinned model")
    subject_range = find_token_range(tokenizer, inputs["input_ids"][0], case["subject"])
    anchor = subject_range[1] - 1
    valid = [
        (row, anchor + offset)
        for row, offset in enumerate(offsets)
        if 0 <= anchor + offset < inputs["input_ids"].shape[1]
    ]
    scores, budget = causal_trace_window_grid(
        model,
        inputs,
        answer_token,
        subject_range,
        layer_paths,
        component="mlp",
        window=10,
        token_indices=[token for _, token in valid],
        config=trace_config,
    )
    aligned = np.full((len(offsets), model.config.n_layer), np.nan)
    aligned[[row for row, _ in valid]] = scores.cpu().numpy()
    aggregate_cases.append(aligned)
    aggregate_passes += budget["model_passes"] + 1
    serializable_scores = [[None if np.isnan(value) else float(value) for value in row] for row in aligned]
    write_json(
        data_dir / f"aggregate-case-{case['known_id']}.json",
        {
            "known_id": case["known_id"],
            "subject_range": list(subject_range),
            "offsets": offsets,
            "scores": serializable_scores,
            "model_passes": budget["model_passes"] + 1,
        },
    )
aggregate_heatmap = np.nanmean(np.stack(aggregate_cases), axis=0)
aggregate_artifact = {
    "case_count": len(aggregate_cases),
    "offsets": offsets,
    "scores": aggregate_heatmap.tolist(),
    "model_passes": aggregate_passes,
}
write_json(data_dir / "aggregate-mlp-heatmap.json", aggregate_artifact)

## Rank-one edit and CounterFact evaluation

`execute_rome` computes the official left/right vectors while restoring the original parameter before returning. TDHook then exposes the rank-one-edited parameter only while the official evaluator runs. Each case records a bitwise restoration check and exact model-pass accounting. The no-edit evaluation is stored separately from the edited evaluation.

In [ ]:
hparams = ROMEHyperParams.from_json(Path(sys.modules["rome"].__file__).parents[1] / "hparams/ROME/gpt2-xl.json")
per_case = []
for record in cases:
    request = record["requested_rewrite"]
    derivation_passes = [0]
    counter_handle = model.register_forward_pre_hook(
        lambda _module, _args: derivation_passes.__setitem__(0, derivation_passes[0] + 1)
    )
    try:
        deltas = execute_rome(model, tokenizer, request, hparams)
    finally:
        counter_handle.remove()
    assert len(deltas) == 1
    weight_name, (left, right) = next(iter(deltas.items()))
    module_path, parameter_name = weight_name.rsplit(".", 1)
    original = nethook.get_parameter(model, weight_name).detach().clone()
    no_edit = compute_rewrite_quality_counterfact(model, tokenizer, record, None, None)
    with temporary_rank_one_edit(model, module_path, parameter_name, left, right) as edited_model:
        edited = compute_rewrite_quality_counterfact(edited_model, tokenizer, record, None, None)
    restored = torch.equal(nethook.get_parameter(model, weight_name), original)
    update = upd_matrix_match_shape(left.unsqueeze(1) @ right.unsqueeze(0), original.shape)
    with torch.no_grad():
        nethook.get_parameter(model, weight_name)[...] += update
    official_edited = compute_rewrite_quality_counterfact(model, tokenizer, record, None, None)
    with torch.no_grad():
        nethook.get_parameter(model, weight_name)[...] = original
    official_application_matches = edited == official_edited
    item = {
        "case_id": record["case_id"],
        "pre": no_edit,
        "post": edited,
        "official_post": official_edited,
        "temporary_edit_restored": restored and torch.equal(nethook.get_parameter(model, weight_name), original),
        "official_application_matches": official_application_matches,
        "model_passes": {"edit_derivation": derivation_passes[0], "evaluation": 3, "total": derivation_passes[0] + 3},
    }
    write_json(data_dir / f"counterfact-case-{record['case_id']}.json", item)
    per_case.append(item)
assert all(item["temporary_edit_restored"] and item["official_application_matches"] for item in per_case)
counterfact_summary = summarize_counterfact(per_case, seed=protocol["counterfact"]["seed"])
official_counterfact_summary = summarize_counterfact(
    [{"post": item["official_post"]} for item in per_case], seed=protocol["counterfact"]["seed"]
)

## Decision and machine-readable summary

The localization gate compares subject-last-token restoration in the middle third against early and late thirds. The CounterFact gate requires every TDHook confidence interval to contain the point estimate produced by an independent official-application run of the same preregistered cases. Until that reference is supplied and every gate is true, the terminal status is `negative_or_incomplete`, never `reproduced`.

In [ ]:
subject_last = aggregate_heatmap[offsets.index(0)]
thirds = np.array_split(subject_last, 3)
localization = bool(thirds[1].mean() > max(thirds[0].mean(), thirds[2].mean()))
gates = {
    "causal_trace_case_parity": all(
        item["parity"]["matches"] and item["mlp_parity"]["matches"] for item in trace_cases
    ),
    "localization": localization,
    "state_restoration": all(item["temporary_edit_restored"] for item in per_case),
    "counterfact_reproduction": all(
        counterfact_summary["metrics"][name]["bootstrap_95_ci"][0]
        <= official_counterfact_summary["metrics"][name]["mean"]
        <= counterfact_summary["metrics"][name]["bootstrap_95_ci"][1]
        for name in protocol["metrics"]
    ),
}
summary = {
    "protocol": protocol,
    "trace_config": config_dict(trace_config),
    "trace_cases": [
        {key: value for key, value in item.items() if not key.endswith("_scores")} for item in trace_cases
    ],
    "aggregate_heatmap": aggregate_artifact,
    "counterfact": counterfact_summary,
    "official_counterfact": official_counterfact_summary,
    "gates": gates,
    "status": "reproduced" if all(gates.values()) else "negative_or_incomplete",
}
summary_sha256 = write_json(data_dir / "summary.json", summary)
print(summary["status"], summary_sha256, gates)